### Train and Test Set Spilt

In [ ]:
import os
import shutil
import random
import sys

# Define paths
IMAGES_SOURCE_DIR = 'D:/Project_P1/data/images0/images0'
LABELS_SOURCE_DIR = 'D:/Project_P1/data/labels0'
TARGET_ROOT_DIR = 'D:/Project_P1/data/airbirds'

TRAIN_RATIO = 0.8
random.seed(42)

def create_yolo_dataset_structure(root_dir):
    os.makedirs(os.path.join(root_dir, 'images', 'train'), exist_ok=True)
    os.makedirs(os.path.join(root_dir, 'images', 'val'), exist_ok=True)
    os.makedirs(os.path.join(root_dir, 'labels', 'train'), exist_ok=True)
    os.makedirs(os.path.join(root_dir, 'labels', 'val'), exist_ok=True)

def print_progress(current, total, prefix=""):
    bar_len = 40
    filled_len = int(round(bar_len * current / float(total)))
    percents = round(100.0 * current / float(total), 1)
    bar = '█' * filled_len + '-' * (bar_len - filled_len)
    sys.stdout.write(f'\r{prefix} |{bar}| {percents}% ({current}/{total})')
    sys.stdout.flush()
    if current == total:
        print()  # New line at the end

def copy_files(image_list, split, images_dir, labels_dir, target_root):
    copied = 0
    skipped = 0
    total = len(image_list)

    for idx, img_name in enumerate(image_list, 1):
        img_src = os.path.join(images_dir, img_name)
        lbl_src = os.path.join(labels_dir, os.path.splitext(img_name)[0] + '.txt')

        if os.path.exists(lbl_src):
            img_dest = os.path.join(target_root, 'images', split, img_name)
            lbl_dest = os.path.join(target_root, 'labels', split, os.path.basename(lbl_src))
            shutil.copy(img_src, img_dest)
            shutil.copy(lbl_src, lbl_dest)
            copied += 1
        else:
            skipped += 1
        print_progress(idx, total, prefix=f"Copying {split} set")

    return copied, skipped

def split_and_copy_files(images_dir, labels_dir, target_root, ratio):
    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(all_images)

    num_train = int(len(all_images) * ratio)
    train_images = all_images[:num_train]
    val_images = all_images[num_train:]

    print(f"🔎 Total images: {len(all_images)}")
    print(f"📂 Training: {len(train_images)}")
    print(f"📂 Validation: {len(val_images)}\n")

    train_copied, train_skipped = copy_files(train_images, 'train', images_dir, labels_dir, target_root)
    val_copied, val_skipped = copy_files(val_images, 'val', images_dir, labels_dir, target_root)

    print("\n✅ Copy complete!")
    print(f"   → Training copied: {train_copied}, skipped: {train_skipped}")
    print(f"   → Validation copied: {val_copied}, skipped: {val_skipped}")

if __name__ == "__main__":
    create_yolo_dataset_structure(TARGET_ROOT_DIR)
    split_and_copy_files(IMAGES_SOURCE_DIR, LABELS_SOURCE_DIR, TARGET_ROOT_DIR, TRAIN_RATIO)


🔎 Total images: 10000
📂 Training: 8000
📂 Validation: 2000

Copying train set |████████████████████████████████████████| 100.0% (8000/8000)
Copying val set |████████████████████████████████████████| 100.0% (2000/2000)

✅ Copy complete!
   → Training copied: 8000, skipped: 0
   → Validation copied: 2000, skipped: 0


### SEQUENCE FORMER FOR LSTM


In [ ]:
import os
from datetime import datetime
import pandas as pd

# ===== CONFIGURATION =====
images_dir = "D:/Project_P1/data/images1"
labels_dir = "D:/Project_P1/data/labels1"

# Define the time gap (in seconds) that signifies a new sequence
time_gap_threshold = 2  # e.g., 5 seconds between images = new sequence

# Define the structure of your filename for parsing
# Example: "D01_20210225093739_0260840"
filename_parts = {
    'prefix_end': 4,    # Length of "D01_" is 4 characters
    'timestamp_start': 4, # Timestamp starts at index 4
    'timestamp_length': 14, # Timestamp is 14 digits (YYYYMMDDHHMMSS)
    # The rest after index 18 (4+14) is the frame number and extension
}

# ===== SCRIPT =====
def parse_timestamp_from_filename(filename):
    """
    Extracts the timestamp from a filename and converts it to a datetime object.
    """
    try:
        # Extract the timestamp substring
        ts_str = filename[filename_parts['timestamp_start']:filename_parts['timestamp_start'] + filename_parts['timestamp_length']]
        # Convert to datetime object
        return datetime.strptime(ts_str, "%Y%m%d%H%M%S")
    except ValueError as e:
        print(f"Error parsing timestamp from filename {filename}: {e}")
        return None

print("Loading and sorting label file data...")
all_file_data = []

# Get all label files
for label_file in os.listdir(labels_dir):
    if label_file.endswith('.txt'):
        file_path = os.path.join(labels_dir, label_file)
        # Parse the timestamp from the filename
        img_timestamp = parse_timestamp_from_filename(label_file)
        
        if img_timestamp is not None:
            # Read the contents of the label file
            with open(file_path, 'r') as f:
                lines = f.readlines()
            
            # For each bird in the image, add an entry
            for line in lines:
                class_id, cx, cy, w, h = map(float, line.strip().split())
                all_file_data.append({
                    'timestamp': img_timestamp,
                    'filename': label_file,
                    'class_id': int(class_id),
                    'center_x': cx,
                    'center_y': cy,
                    'width': w,
                    'height': h
                })
        else:
            print(f"Skipping file due to timestamp error: {label_file}")

# Create DataFrame and sort by timestamp
df = pd.DataFrame(all_file_data)
df.sort_values('timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

print("Grouping into discrete sequences based on time gap...")
# Calculate time difference between consecutive entries
df['time_diff'] = df['timestamp'].diff().dt.total_seconds()

# A gap larger than our threshold marks the start of a new sequence
df['sequence_id'] = (df['time_diff'] > time_gap_threshold).cumsum()

# Get the number of sequences found
num_sequences = df['sequence_id'].nunique()
print(f"Found {num_sequences} discrete sequences.")

# Analyze each sequence
sequences = []
for seq_id, group in df.groupby('sequence_id'):
    num_frames = group['filename'].nunique() # Frames in this sequence
    num_birds = len(group)                   # Total bird detections
    duration = (group['timestamp'].max() - group['timestamp'].min()).total_seconds()
    
    sequences.append({
        'sequence_id': seq_id,
        'start_time': group['timestamp'].min(),
        'end_time': group['timestamp'].max(),
        'duration_seconds': duration,
        'number_of_frames': num_frames,
        'number_of_detections': num_birds,
        'avg_birds_per_frame': num_birds / num_frames
    })


# Save the full data with sequence IDs
full_data_path = "D:/Project_P1/data/full_detections_with_sequence_id.csv"
df.to_csv(full_data_path, index=False)
print(f"Full data with sequence IDs saved to: {full_data_path}")

print("\nYou can now analyze each sequence independently by filtering on the 'sequence_id' column.")

Loading and sorting label file data...
Grouping into discrete sequences based on time gap...
Found 10 discrete sequences.
Full data with sequence IDs saved to: D:/Project_P1/data/full_detections_with_sequence_id.csv

You can now analyze each sequence independently by filtering on the 'sequence_id' column.


### To extract sequence from a Dataset from kaggle


In [1]:
import os
import glob
import xml.etree.ElementTree as ET
import pandas as pd

# Path where your XML annotation files are stored
# Example: "D:/Project_P1/data/annotations"
ANNOTATIONS_DIR = r"C:\Users\visha\Downloads\archive\FBD-SV-2024\labels\train"

data_rows = []

# Iterate over all XML files in subfolders
for xml_file in glob.glob(os.path.join(ANNOTATIONS_DIR, "**", "*.xml"), recursive=True):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Extract filename (e.g. bird_100_000019.jpg)
    filename = root.find("filename").text   
    
    # Split by "_" and take first two parts → video name
    parts = filename.split("_")  # ["bird", "100", "000019.jpg"]
    video_id = "_".join(parts[:2])  # "bird_100"
    
    # Get frame number from filename suffix (000019 → 19)
    frame_no = int(os.path.splitext(parts[-1])[0])
    
    # Get bounding box
    for obj in root.findall("object"):
        bbox = obj.find("bndbox")
        x1 = int(bbox.find("xmin").text)
        y1 = int(bbox.find("ymin").text)
        x2 = int(bbox.find("xmax").text)
        y2 = int(bbox.find("ymax").text)
        
        data_rows.append([video_id, x1, y1, x2, y2, frame_no])

# Save to DataFrame
df = pd.DataFrame(data_rows, columns=["video", "x1", "y1", "x2", "y2", "frame"])

# Sort by video and frame to keep proper sequence
df.sort_values(by=["video", "frame"], inplace=True)

# Save to CSV
output_csv = "bird_sequences.csv"
df.to_csv(output_csv, index=False)

print(f"CSV file saved as {output_csv}")

CSV file saved as bird_sequences.csv


In [1]:
import csv
import re

def convert_text_to_csv(input_file_name, output_file_name):
    """
    Reads a raw text file with tab-separated bird observation data,
    parses it, and writes it to a clean CSV file.

    Args:
        input_file_name (str): The full path to the input text file.
        output_file_name (str): The full path for the output CSV file.
    """
    try:
        with open(input_file_name, 'r', encoding='utf-8') as f:
            content = f.read()
        
        try:
            header, body = content.split('\n', 1)
        except ValueError:
            print("\nError: The input file appears to be empty or has no header line.")
            return
            
        header_columns = header.strip().split('\t')

        records = re.split(r'(?=URN:)', body)
        cleaned_records = [rec.strip() for rec in records if rec.strip()]

        # --- Debug print to verify the first record ---
        print("--- Verifying the first data record found ---")
        if cleaned_records:
            print(cleaned_records[0])
        else:
            print("Warning: No data records starting with 'URN:' were found.")
        print("---------------------------------------------")

        with open(output_file_name, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(header_columns)
            
            for record in cleaned_records:
                row_data = record.split('\t')
                writer.writerow(row_data)

        print(f"\n✅ Process complete.")
        print(f"Output saved to: {output_file_name}")

    except FileNotFoundError:
        print(f"Error: The file '{input_file_name}' was not found.")
        print("Please make sure the path is correct and the file exists.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# --- Main execution ---
if __name__ == "__main__":
    # Use your specific file paths
    input_filename = "D:/Project_P1/data/ebd_IN-TN-CO_smp_relJun-2025.txt"
    output_filename = "D:/Project_P1/data/ebird_observations.csv"
    
    convert_text_to_csv(input_filename, output_filename)

--- Verifying the first data record found ---
URN:CornellLabOfOrnithology:EBIRD:OBS998276745	2023-04-22 16:05:29.051171	3219	species	avibase-61503588	Asian Koel	Eudynamys scolopaceus				X					India	IN	Tamil Nadu	IN-TN	Coimbatore	IN-TN-CO					kurumbapalayam	L5279726	P	11.1115221	77.0210719	1966-04-24		obsr472391		S75023094	Historical	Historical	P62						1	1		0	1	0		ஒரு ஜோடி தையற்காரப் பறவை காதல் செய்வதைப் பார்த்தேன். வேறு சில பறவைகளும் தென்பட்டன. தலை வெளிர் நீல நிறமுடைய minivet  போன்ற பறவையைக் கண்டேன்.--மழையில் ஒரு reef heron தென்னை மரத்திலிருந்து கூரை மீது விழுந்து வாசலில் தண்ணீரில் வந்து விழுந்தது. அதை இரவெல்லாம் கூடையில் மூடி வைத்திருந்து காலையில் அதற்கு ஒரு ring  போட்டுப் பறக்க விட்டேன். இப்படி ring  போட இயலும் என ஒரு நாளும் நான் கனவிலும் நினைக்கவில்லை. --வீட்டுக்குப் பின்னாலிருந்த மொட்டைக் கமுக மரத்தை மைனாக்கள் வைத்துக் கொண்டிருந்தன. --(ரகிடு) shikra  வேப்பமரத்தில் ஒருமணி நேரம் ஐந்து முறை ஒரு அணிலை திரும்பத் திரும்ப பிடிக்க முயன்று தோல்வி அடைந்ததைப் பார்த்தபடி பாதாம் மரத்தடியில் பட

## To lighten the model

In [ ]:
import tensorflow as tf
import os

# --- Configuration ---
# Set the path to your trained Keras model
SAVED_MODEL_PATH = "D:/Project_P1/dev/universal_bird_predictor.h5"
# Set the desired output path for the TFLite model
TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"

print(f"Loading Keras model from: {SAVED_MODEL_PATH}")

# Load the trained Keras model
try:
    model = tf.keras.models.load_model(SAVED_MODEL_PATH, compile=False)
except IOError:
    print(f"ERROR: Could not find the Keras model file at '{SAVED_MODEL_PATH}'. Please check the path.")
    exit()

# Create a TFLite converter object from the Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# --- FIX: Add settings to handle LSTM conversion error ---
# This allows the converter to use select TensorFlow operations when a native
# TFLite operation is not available, which is needed for certain LSTM behaviors.
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,  # Enable TFLite ops.
    tf.lite.OpsSet.SELECT_TF_OPS    # Enable select TensorFlow ops.
]
converter._experimental_lower_tensor_list_ops = False
# ---------------------------------------------------------

# Apply default optimizations (includes quantization which can speed up inference)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

print("Converting the model to TensorFlow Lite format...")
# Convert the model
tflite_model = converter.convert()

print(f"Saving TFLite model to: {TFLITE_MODEL_PATH}")
# Save the converted model to a .tflite file
with open(TFLITE_MODEL_PATH, 'wb') as f:
    f.write(tflite_model)

print("\n✅ Success! Model conversion is complete.")
print(f"The optimized model has been saved to '{TFLITE_MODEL_PATH}'.")
print("You can now use this file in the main prediction script.")

Loading Keras model from: D:/Project_P1/dev/universal_bird_predictor.h5


AttributeError: module 'tensorflow' has no attribute 'keras'